# Day 47: Visualize PagedAttention Block Layout

**Layer:** Implementation


## Concept Overview

PagedAttention's block table maps logical token positions to non-contiguous physical memory blocks, enabling dynamic allocation and copy-on-write prefix sharing. Visualizing the block layout reveals how memory is used and where fragmentation occurs.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. PagedAttention Block Layout Visualization

PagedAttention maps logical sequence positions to physical memory blocks via a block table. This visualization shows how blocks are allocated, reused, and freed.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def visualize_paged_memory(sequences, block_size=16, total_blocks=32):
    """Visualize PagedAttention block allocation."""
    # Assign blocks to sequences
    block_table = {}
    free_blocks = list(range(total_blocks))
    seq_colors = plt.cm.tab10(np.linspace(0, 1, len(sequences)))

    for i, (seq_id, seq_len) in enumerate(sequences):
        n_blocks = (seq_len + block_size - 1) // block_size
        blocks = free_blocks[:n_blocks]
        free_blocks = free_blocks[n_blocks:]
        block_table[seq_id] = (blocks, seq_len, seq_colors[i])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

    # Physical memory view
    for i in range(total_blocks):
        owner = None
        for seq_id, (blocks, seq_len, color) in block_table.items():
            if i in blocks:
                owner = (seq_id, seq_len, color)
                break
        color = owner[2] if owner else 'lightgray'
        rect = plt.Rectangle((i, 0), 1, 1, color=color, edgecolor='black')
        ax1.add_patch(rect)
        ax1.text(i+0.5, 0.5, f'B{i}', ha='center', va='center', fontsize=7)
    ax1.set_xlim(0, total_blocks); ax1.set_ylim(0, 1)
    ax1.set_title('Physical GPU Memory: Block Allocation')
    ax1.set_xlabel('Physical Block ID'); ax1.set_yticks([])

    # Logical view: sequence → blocks mapping
    for i, (seq_id, (blocks, seq_len, color)) in enumerate(block_table.items()):
        for j, bid in enumerate(blocks):
            tokens_in_block = min(block_size, seq_len - j*block_size)
            fill_pct = tokens_in_block / block_size
            rect = plt.Rectangle((j, -i-0.9), fill_pct, 0.8, color=color, alpha=0.8)
            ax2.add_patch(rect)
            rect2 = plt.Rectangle((j, -i-0.9), 1, 0.8, fill=False, edgecolor='black')
            ax2.add_patch(rect2)
        ax2.text(-0.5, -i-0.5, f'Seq {seq_id} ({seq_len}t)', ha='right', va='center', fontsize=9)
    ax2.set_xlim(-1, max(len(v[0]) for v in block_table.values())+1)
    ax2.set_ylim(-len(sequences)-0.5, 0.5)
    ax2.set_title('Logical Block Table: Sequence → Physical Blocks')
    ax2.set_xlabel('Logical Block Index'); ax2.set_yticks([])

    plt.tight_layout()
    plt.savefig('paged_attention_layout.png', dpi=100, bbox_inches='tight')
    plt.show()

sequences = [('A', 48), ('B', 80), ('C', 24), ('D', 96), ('E', 16)]
visualize_paged_memory(sequences)


## 2. Fragmentation Analysis

Block-level allocation introduces internal fragmentation: the last block of each sequence may be partially filled.


In [ ]:
def analyze_fragmentation(seq_lens, block_size=16):
    total_tokens = sum(seq_lens)
    total_blocks = sum((l+block_size-1)//block_size for l in seq_lens)
    allocated_tokens = total_blocks * block_size
    waste = allocated_tokens - total_tokens
    return waste / allocated_tokens

print('Fragmentation analysis:')
for block_size in [8, 16, 32, 64]:
    np.random.seed(42)
    seq_lens = np.random.randint(10, 200, 100).tolist()
    frag = analyze_fragmentation(seq_lens, block_size)
    print(f'  Block size={block_size:>3}: fragmentation={frag:.1%}')
print('Smaller blocks = less fragmentation, but more overhead')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- PagedAttention's block table maps logical token positions to non-contiguous physical memory blocks, enabling dynamic allocation and copy-on-write prefix sharing.
- Block size is a key PagedAttention hyperparameter: small blocks (8 tokens) minimize fragmentation but add metadata overhead; large blocks (128 tokens) are efficient for long sequences but waste memory for short ones..
- Day 47 implementation complete.
